In [48]:
import pandas as pd
from pathlib import Path
import logging
import os
import re
from datetime import datetime

In [49]:
root_folder = Path.cwd()
print(root_folder)

c:\Users\Aryan\OneDrive\Desktop\internship


In [50]:
file_path = root_folder/"dirty_dataset.csv"
original_df = pd.read_csv(file_path)
df = original_df.copy()
df.head()

,discount_pct,unit_price,quantity,order_date,total_amount,customer_id,product_name,pincode,status,rating,order_id,city,category,payment_method,phone,state
0,20,Rs. 120.00,NaN,2023-09-26,-2280.00,CUST_0544,HDMI Cable,499100,Cancelled,Good,ORD_01,NaN,electronics,credit card,9782111511,tamil nadu
1,NaN,750#,4,26/09/2023,Rs.622,00131,Keyboard (New),380320,P3nding,0,ORD_02,PUNE,Electronicss,WALLET,919271224104,NaN
2,NaN,199,4,17-01-2024,NaN,CUST_,Phone Holder,60638,Delivered,5 stars,ord_0003,unknown,Accessories,NaN,919507469456,Maharashtra
3,NaN,inf,9999,1734805800,0.00,CUST_???,Desk Lamp,072987,pending,3,ORD_0004,unknown,electronics,CREDIT CARD,91737 60396,karnataka
4,-5,Rs. 650.00,NaN,05/09/2024,$520.00,CUST_0187,Desk Lamp,71071,Canc,NaN,ORD_0005,"Chennai,",Electronics,NaN,+91-93885-07486,Tamil Nadu


##### 1. Checking the shape of the data

In [51]:

print(f"The no. of rows in the dataset = {df.shape[0]}")
print(f"The no. of columns  in the dataset = {df.shape[1]}")

The no. of rows in the dataset = 1064
The no. of columns  in the dataset = 16


##### 2. Checking the info of the data 

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1064 entries, 0 to 1063
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   discount_pct    734 non-null    object
 1   unit_price      952 non-null    object
 2   quantity        833 non-null    object
 3   order_date      1060 non-null   object
 4   total_amount    963 non-null    object
 5   customer_id     957 non-null    object
 6   product_name    1060 non-null   object
 7   pincode         804 non-null    object
 8   status          1059 non-null   object
 9   rating          819 non-null    object
 10  order_id        1060 non-null   object
 11  city            906 non-null    object
 12  category        1058 non-null   object
 13  payment_method  740 non-null    object
 14  phone           900 non-null    object
 15  state           764 non-null    object
dtypes: object(16)
memory usage: 133.1+ KB


##### 2.1) Checking the order_date 

In [ ]:
import logging
from logging.handlers import RotatingFileHandler

log_dir = os.path.join("logging","logging_ingestion")
def get_logger():
    os.makedirs(log_dir,exist_ok = True)
    log_file = os.path.join(log_dir,"ingesiton_log.log")
    logger = logging.getLogger("ingestion")

    if logger.handlers:
        return logger

    logger.setLevel(logging.INFO)

    handler = RotatingFileHandler(
        filename = log_file,
        mode = 'a',
        maxBytes = 5*1024*1024,
        backupCount = 10
    )

    handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
        )
    )

    logger.addHandler(handler)
    return logger


logger = get_logger()   

In [ ]:

QUARANTINE_DIR = os.path.join("quarantine", "quarantine_date")

from datetime import datetime
ist_offset = pd.Timedelta(hours = 5,minutes = 30)
def _parse_date(val):
    date_formats= [
        "%Y-%m-%d", # 2023-09-2026
        "%d/%m/%Y", # 26/09/2023
        "%d-%m-%Y", # 17-01-2024
        "%b %d, %Y",# Sep 20, 2023
        "%d-%b-%Y", # 13-Jun-2024
    ]

    if pd.isnull(val) or str(val).strip()=="":
        return pd.NaT
    
    if val.isdigit():
        return pd.Timestamp(int(val),unit='s') + ist_offset
    
    for fmt in date_formats:
        try:
            return datetime.strptime(val,fmt)
        except ValueError:
            continue
    return val

def clean_date(df, original_df):
    

    
    logger.info("Starting clean_date process...")
    df['order_date_parsed'] = df['order_date'].apply(_parse_date)

    unparsed_mask  = df['order_date_parsed'].apply(lambda x: isinstance(x, str))
    failed_indexes = df[unparsed_mask].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_date: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
    else: 
        logger.info("clean_date: All rows parsed successfully.") 

   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['order_date'])
    clean_df = clean_df.rename(columns={'order_date_parsed': 'order_date'})

    clean_df['order_date'] = pd.to_datetime(
        clean_df['order_date'], errors='coerce'
    )

    logger.info("clean_date process completed.")
    return clean_df, failed_indexes








UNIT PRICE

In [ ]:
def _parse_price(val):
    if pd.isnull(val) or str(val).strip()=="":
        return pd.NA
    val = str(val).strip()

    if val.lower()=='inf':
        return pd.NA

    val = val.replace("Rs.","").replace("$","").strip()
    val = val.replace(",","")
    val = val.rstrip("#").rstrip(".").strip()

    try:
        return float(val)
    except ValueError:
        return val
    

def clean_price(df, original_df):
    logger.info("Starting clean_discount process...")

    
    df['unit_price_clean'] = df['unit_price'].apply(_parse_price)
    unparsed_price = df['unit_price_clean'].apply(lambda x:isinstance(x,str))
    
    failed_indexes = df[unparsed_price].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_discount: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_discount: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['unit_price'])
    clean_df = clean_df.rename(columns={'unit_price_clean': 'unit_price'})

    clean_df['unit_price'] = pd.to_datetime(
        clean_df['unit_price'], errors='coerce'
    )

    logger.info("clean_discount process completed.")
    return clean_df, failed_indexes




DISCOUNT

In [ ]:
def _parse_discount(val):
    if pd.isnull(val) or str(val).strip()=="":
        return pd.NA
    
    val = str(val).strip()
    val = val.replace("%","").replace("_dup","")

    if str(val) == "_dup":
        return pd.NA

    try:
        return float(val)
    except ValueError:
        return val
    
def clean_discount(df, original_df):
    

    
    df['discount_clean'] = df['discount_pct'].apply(_parse_discount)
    unparsed_discount = df['discount_clean'].apply(lambda x : isinstance(x,str))
    
    failed_indexes = df[unparsed_discount].index.tolist()

    
   
    if failed_indexes:

        logger.info("Starting clean_quantity process...")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
    else:
        logger.info("clean_quantity: All rows parsed successfully.")  

   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['discount_pct'])
    clean_df = clean_df.rename(columns={'discount_clean': 'discount_pct'})

    clean_df['discount_pct'] = pd.to_datetime(
        clean_df['discount_pct'], errors='coerce'
    )


    logger.info("clean_amount process completed.")
    return clean_df, failed_indexes




TOTAL AMOUNT

In [ ]:
def _parse_total_amount(val):
    
    if pd.isnull(val) or str(val).strip() == "" or str(val).lower() in ['nan', 'null', 'n/a']:
        return pd.NA
    
    val = str(val).strip()
    
   
    if val.lower() == 'inf':
        return pd.NA
    
  
    val = val.replace("Rs.", "").replace("$", "")
    val = val.replace(",", "")
    

    
    val = val.strip()
    
    try:
        return float(val)
    except ValueError:
        return val
    

def clean_amount(df, original_df):
    logger.info("Starting clean_product_name process...")

    
    df['total_amount_clean'] = df['total_amount'].apply(_parse_total_amount)


    unparsed_total = df['total_amount_clean'].apply(lambda x: isinstance(x, str))
    
    failed_indexes = df[unparsed_total].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_amount: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")

        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       

    else:
        logger.info("clean_amount: All rows parsed successfully.")
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['total_amount'])
    clean_df = clean_df.rename(columns={'total_amount_clean': 'total_amount'})

    clean_df['total_amount'] = pd.to_datetime(
        clean_df['total_amount'], errors='coerce'
    )

    logger.info("clean_amount process completed.")
    return clean_df, failed_indexes



QUANTITY

In [ ]:


def _parse_quantity(val):

    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() in ["nan", "null", "n/a", "<na>"]
    ):
        return pd.NA

    val = str(val).strip()

    val = val.replace("-", "")

    try:
       
        num = int(float(val))

       
        if num == 9999:
            return pd.NA

        return num
    except ValueError:
        return val

def clean_quantity(df, original_df):
    logger.info("Starting clean_quantity process...")
    

    df['quantity_clean'] = df['quantity'].apply(_parse_quantity)


    unparsed_quantity = df['quantity_clean'].apply(lambda x: isinstance(x, str))
    
    failed_indexes = df[unparsed_quantity].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_quantity: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)

    else:
        logger.info("clean_quantity: All rows parsed successfully.") 

   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['quantity'])
    clean_df = clean_df.rename(columns={'quantity_clean': 'quantity'})

    clean_df['quantity'] = pd.to_datetime(
        clean_df['quantity'], errors='coerce'
    )


    
    logger.info("clean_quantity process completed.")
    return clean_df, failed_indexes







PRODUCT NAME

In [ ]:
def _parse_product_name(val):
    
    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() in ["nan", "null", "n/a", "<na>"]
        ):
        return pd.NA

    val = str(val).strip()

    val = val.replace(" (New)", "").replace(" (New )", "")
    val = val.replace(" - Imported", "")
    val = val.replace(", Pack of 3", "")
    val = val.strip()  # Ek baar fir safe side ke liye strip


    
    return val
    
def clean_product_name(df, original_df):
    
    logger.info("Starting clean_product_name process...")
    
    df["product_name_clean"] = df['product_name'].apply(_parse_product_name)

    unparsed_product_name = df["product_name_clean"].apply(lambda x: isinstance(x,str) and x is not pd.NA)
    
    failed_indexes = df[unparsed_product_name].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_product_name: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_product_name: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['product_name'])
    clean_df = clean_df.rename(columns={'product_name_clean': 'product_name'})

    clean_df['product_name'] = pd.to_datetime(
        clean_df['product_name'], errors='coerce'
    )

    logger.info("clean_product_name process completed.")
    return clean_df, failed_indexes




PINCODE

In [ ]:
def _parse_pincode(val):

    
    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() in ["nan", "null", "n/a", "<na>"]
        ):
        return pd.NA
    val = str(val).strip()

    if str(val) == 'DUPLICATE':
        return pd.NA
    

    val = val.rstrip("_dup").rstrip(".").strip()




    try:
        num = int(val)
        return num
    
    except ValueError:
        return val
    
def clean_pincode(df, original_df):
    
    logger.info("Starting clean_pincode process...")
    
    df['pincode_clean'] = df['pincode'].apply(_parse_pincode)
    unparsed_pincode = df['pincode_clean'].apply(lambda x:isinstance(x,str))
    
    failed_indexes = df[unparsed_pincode].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_pincode: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_pincode: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['pincode'])
    clean_df = clean_df.rename(columns={'pincode_clean': 'pincode'})

    clean_df['pincode'] = pd.to_datetime(
        clean_df['pincode'], errors='coerce'
    )
    
    logger.info("clean_pincode process completed.")
    return clean_df, failed_indexes


STATUS

In [ ]:
def _parse_status(val):


    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() == "nan"
    ):
        return pd.NA


    val = str(val).strip().lower()

    val = val.replace("_dup","")


    if (
        "pend" in val
        or "p3nd" in val
        or val == "pending"
        or val == "p3nding"
        or val == "pendind"
    ):
        return "Pending"


    elif "ship" in val or "shipp3d" in val:
        return "Shipped"


    elif (
        "deliv" in val
        or "d3liv" in val
        or val == "delivered"
        or val == "deli"
        or val == "d3liv3r3d"
    ):
        return "Delivered"

    elif "canc" in val or "canc3l" in val:
        return "Cancelled"

    elif "retu" in val or "r3turn" in val:
        return "Returned"

    
    return val.title()

def clean_status(df, original_df):
    logger.info("Starting clean_status process...")
    

    
    df["status_clean"] = df["status"].apply(_parse_status)


    unparsed_status = df["status_clean"].apply(
    lambda x: isinstance(x, str) and x is not pd.NA
)
    
    failed_indexes = df[unparsed_status].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_status: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_status: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['status'])
    clean_df = clean_df.rename(columns={'status_clean': 'status'})

    clean_df['status'] = pd.to_datetime(
        clean_df['status'], errors='coerce'
    )
    logger.info("clean_status process completed.")
    return clean_df, failed_indexes







CITY

In [ ]:
def _parse_city(val):


    if(

        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() in ["nan", "null", "n/a", "<na>","unknown"]
    ):
        return pd.NA
    
    val = str(val).strip().lower()

    val = val.replace(",", "").replace("_dup", "")
    val = val.strip()
    return val.title()


def clean_city(df, original_df):
    
    logger.info("Starting clean_city process...")
    
    df["city_clean"] = df["city"].apply(_parse_city)


    unparsed_city = df["city_clean"].apply(
    lambda x: isinstance(x, str) and x is not pd.NA
)
    
    failed_indexes = df[unparsed_city].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_city: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       

    else:
        logger.info("clean_city: All rows parsed successfully.")
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['city'])
    clean_df = clean_df.rename(columns={'city_clean': 'city'})

    clean_df['city'] = pd.to_datetime(
        clean_df['city'], errors='coerce'
    )
    logger.info("clean_city process completed.")
    return clean_df, failed_indexes





CATEGORY


In [ ]:
import pandas as pd


def _parse_category(val):
    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() in ["nan", "null", "n/a", "<na>"]
    ):
        return pd.NA
    val = str(val).strip().lower()

  


    if "elec" in val:
        return "Electronics"

    elif "acce" in val or "access" in val:
        return "Accessories"

    elif "stat" in val:
        return "Stationery"

    elif "furn" in val:
        return "Furniture"

    return val.title()


def clean_category(df, original_df):
    
    logger.info("Starting clean_category process...")
    
    df["category_clean"] = df["category"].apply(_parse_category)


    unparsed_category = df["category_clean"].apply(
    lambda x: isinstance(x, str) and x is not pd.NA
)
    
    failed_indexes = df[unparsed_category].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_category: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
    
    else:
        logger.info("clean_category: All rows parsed successfully.")

   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['category'])
    clean_df = clean_df.rename(columns={'category_clean': 'category'})

    clean_df['category'] = pd.to_datetime(
        clean_df['category'], errors='coerce'
    )

    logger.info("clean_category process completed.")
    return clean_df, failed_indexes




STATE


In [ ]:

def _parse_state(val):
    
    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() in ["nan", "null", "n/a", "<na>"]
    ):
        return pd.NA

    
    val = str(val).strip().lower()

   
    val = val.replace("@", "a")

    
    if "tamil" in val:
        return "Tamil Nadu"
    elif "maha" in val or "mah" in val:
        return "Maharashtra"
    elif "karn" in val:
        return "Karnataka"
    elif "guj" in val:
        return "Gujarat"
    elif "raj" in val:
        return "Rajasthan"
    elif "beng" in val or "west" in val:
        return "West Bengal"
    elif "tel" in val:
        return "Telangana"
    elif "delh" in val:
        return "Delhi"

    
    return val.title()



def clean_state(df, original_df):

    logger.info("Starting clean_state process...")
    
    df["state_clean"] = df["state"].apply(_parse_state)
    unparsed_state = df["state_clean"].apply(
    lambda x: isinstance(x, str) and x is not pd.NA
)

    
    failed_indexes = df[unparsed_state].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_state: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_state: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['state'])
    clean_df = clean_df.rename(columns={'state_clean': 'state'})

    clean_df['state'] = pd.to_datetime(
        clean_df['state'], errors='coerce'
    )

    logger.info("clean_state process completed.")
    return clean_df, failed_indexes



PAYMENT METHOD

In [ ]:
def _parse_payment(val):

    if (
        pd.isnull(val)
        or str(val).strip() == ""
        or str(val).lower() == ["nan", "null", "n/a", "<na>"]

    ):
        return pd.NA
    val = str(val).strip().lower()

    return val.title()


def clean_payment(df, original_df):
    

    logger.info("Starting clean_payment process...")
    df["payment_clean"] = df["payment_method"].apply(_parse_payment)


    unparsed_payment = df["payment_clean"].apply(
        lambda x: isinstance(x, str) and x is not pd.NA
)
    
    failed_indexes = df[unparsed_payment].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_payment: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_payment: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['payment_method'])
    clean_df = clean_df.rename(columns={'payment_clean': 'payment_method'})

    clean_df['payment_method'] = pd.to_datetime(
        clean_df['payment_method'], errors='coerce'
    )

    logger.info("clean_payment process completed.")
    return clean_df, failed_indexes


PHONE NUMBER

In [ ]:

def _parse_phone(val):
    if pd.isnull(val) or str(val).strip() =="":
        return pd.NA
    val = str(val).strip()
    digits = re.sub(r'\D', '', val)
    if len(digits) == 12 and digits.startswith("91"):
        digits = digits[2:]
    elif len(digits) == 11 and digits.startswith("0"):
        digits= digits[1:]
    if len(digits) == 10:
        return digits
    else:
        return val



    
def clean_phone(df, original_df):
    
    logger.info("Starting clean_phone process...")
    
    df["phone_clean"] = df["phone"].apply(_parse_phone)


    unparsed_phone = df['phone_clean'].apply(lambda x:isinstance(x,str))
    
    failed_indexes = df[unparsed_phone].index.tolist()

    
   
    if failed_indexes:
        logger.warning(f"clean_phone: {len(failed_indexes)} rows failed parsing. Moving to quarantine.")
        os.makedirs(QUARANTINE_DIR, exist_ok=True)

        quarantine_rows = original_df.loc[failed_indexes].copy()
       
        quarantine_path = os.path.join(
            QUARANTINE_DIR,
            f"quarantine_date_{pd.Timestamp.now().strftime('%d-%m-%Y')}.csv"
        )
        quarantine_rows.to_csv(quarantine_path, index=False)
       
    else:
        logger.info("clean_phone: All rows parsed successfully.")
   
    clean_df = df.drop(index=failed_indexes)

    

    clean_df = clean_df.drop(columns=['phone'])
    clean_df = clean_df.rename(columns={'phone_clean': 'phone'})

    clean_df['phone'] = pd.to_datetime(
        clean_df['phone'], errors='coerce'
    )

    logger.info("clean_phone process completed.")
    return clean_df, failed_indexes



df.info()

In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1064 entries, 0 to 1063
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   discount_pct    734 non-null    object
 1   unit_price      952 non-null    object
 2   quantity        833 non-null    object
 3   order_date      1060 non-null   object
 4   total_amount    963 non-null    object
 5   customer_id     957 non-null    object
 6   product_name    1060 non-null   object
 7   pincode         804 non-null    object
 8   status          1059 non-null   object
 9   rating          819 non-null    object
 10  order_id        1060 non-null   object
 11  city            906 non-null    object
 12  category        1058 non-null   object
 13  payment_method  740 non-null    object
 14  phone           900 non-null    object
 15  state           764 non-null    object
dtypes: object(16)
memory usage: 133.1+ KB
